# Part 2 Cell Population Frequencies
What is the frequency of each cell type in each sample?

## Method 
For each sample:
1. Sum 5 immune cell population counts to calculate 'total_count'
2. Divide each population by 'total_count'
3. Multiply by 100 for percent value

In [3]:
from pathlib import Path
import sqlite3

import pandas as pd

DB_PATH = Path("cell-count.db")

if not DB_PATH.exists():
    raise FileNotFoundError("Run `python load_data.py` before opening this notebook.")

## Frequency Table

In [6]:
raw_counts_query = """
SELECT
    counts.sample_id AS sample,
    populations.population_name AS population,
    counts.cell_count AS count
FROM cell_counts AS counts
INNER JOIN cell_populations AS populations
    ON populations.population_id = counts.population_id
ORDER BY counts.sample_id, populations.population_name;
"""

with sqlite3.connect(DB_PATH) as connection:
    raw_counts_df = pd.read_sql_query(raw_counts_query, connection)

raw_counts_df.head()

,sample,population,count
0,sample00000,b_cell,10908
1,sample00000,cd4_t_cell,20491
2,sample00000,cd8_t_cell,24440
3,sample00000,monocyte,23511
4,sample00000,nk_cell,13864


## Calculate Frequencies
Calculated total cell count per sample then each count converted to percentage of sample total

In [8]:
frequency_df = raw_counts_df.copy()
frequency_df["total_count"] = frequency_df.groupby("sample")["count"].transform("sum")
frequency_df["percentage"] = (
    100 * frequency_df["count"] / frequency_df["total_count"]
).round(4)

frequency_df = frequency_df[
    ["sample", "total_count", "population", "count", "percentage"]
]

frequency_df

,sample,total_count,population,count,percentage
0,sample00000,93214,b_cell,10908,11.7021
1,sample00000,93214,cd4_t_cell,20491,21.9827
2,sample00000,93214,cd8_t_cell,24440,26.2192
3,sample00000,93214,monocyte,23511,25.2226
4,sample00000,93214,nk_cell,13864,14.8733
...,...,...,...,...,...
52495,sample10499,92418,b_cell,9258,10.0175
52496,sample10499,92418,cd4_t_cell,26769,28.9651
52497,sample10499,92418,cd8_t_cell,29100,31.4874
52498,sample10499,92418,monocyte,14396,15.5771


## Preview
Full result has 5 rows for each sample one for each cell population

In [9]:
frequency_df.head()

,sample,total_count,population,count,percentage
0,sample00000,93214,b_cell,10908,11.7021
1,sample00000,93214,cd4_t_cell,20491,21.9827
2,sample00000,93214,cd8_t_cell,24440,26.2192
3,sample00000,93214,monocyte,23511,25.2226
4,sample00000,93214,nk_cell,13864,14.8733
